In [2]:
import sys
print(sys.executable)

/Users/christianormsby/Documents/premier-league-performance-intelligence/.venv/bin/python3.13


In [3]:
import pandas as pd

In [19]:
df = pd.read_csv("../data/raw/E0.csv")

In [20]:
df.shape

(380, 132)

In [6]:
df.columns.tolist()

['Div',
 'Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR',
 'B365H',
 'B365D',
 'B365A',
 'BFDH',
 'BFDD',
 'BFDA',
 'BMGMH',
 'BMGMD',
 'BMGMA',
 'BVH',
 'BVD',
 'BVA',
 'BWH',
 'BWD',
 'BWA',
 'CLH',
 'CLD',
 'CLA',
 'LBH',
 'LBD',
 'LBA',
 'PSH',
 'PSD',
 'PSA',
 'MaxH',
 'MaxD',
 'MaxA',
 'AvgH',
 'AvgD',
 'AvgA',
 'BFEH',
 'BFED',
 'BFEA',
 'B365>2.5',
 'B365<2.5',
 'P>2.5',
 'P<2.5',
 'Max>2.5',
 'Max<2.5',
 'Avg>2.5',
 'Avg<2.5',
 'BFE>2.5',
 'BFE<2.5',
 'AHh',
 'B365AHH',
 'B365AHA',
 'PAHH',
 'PAHA',
 'MaxAHH',
 'MaxAHA',
 'AvgAHH',
 'AvgAHA',
 'BFEAHH',
 'BFEAHA',
 'B365CH',
 'B365CD',
 'B365CA',
 'BFDCH',
 'BFDCD',
 'BFDCA',
 'BMGMCH',
 'BMGMCD',
 'BMGMCA',
 'BVCH',
 'BVCD',
 'BVCA',
 'BWCH',
 'BWCD',
 'BWCA',
 'CLCH',
 'CLCD',
 'CLCA',
 'LBCH',
 'LBCD',
 'LBCA',
 'PSCH',
 'PSCD',
 'PSCA',
 'MaxCH',
 'MaxCD',
 'MaxCA',
 'AvgCH',
 'Avg

In [21]:
home = df[[
    'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR'
]].copy()

home.columns = [
    'Date', 'Team', 'Opponent',
    'Goals', 'GoalsConceded', 'FTR'
]

away = df[[
    'Date', 'AwayTeam', 'HomeTeam',
    'FTAG', 'FTHG', 'FTR'
]].copy()

away.columns = [
    'Date', 'Team', 'Opponent',
    'Goals', 'GoalsConceded', 'FTR'
]

print(home.shape)
print(away.shape)

(380, 6)
(380, 6)


In [22]:
team_matches = pd.concat(
    [home, away],
    ignore_index=True
)

team_matches.shape

(760, 6)

In [7]:
df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [24]:
def get_result(row):
    if row['Goals'] > row['GoalsConceded']:
        return 'W'
    elif row['Goals'] < row['GoalsConceded']:
        return 'L'
    else:
        return 'D'

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Columns: 132 entries, Div to BFECAHA
dtypes: float64(108), int64(16), str(8)
memory usage: 392.0 KB


In [25]:
team_matches['Result'] = team_matches.apply(
    get_result,
    axis=1
)

team_matches['Result'].value_counts()

Result
W    276
L    276
D    208
Name: count, dtype: int64

In [9]:
df.dtypes

Div             str
Date            str
Time            str
HomeTeam        str
AwayTeam        str
             ...   
MaxCAHA     float64
AvgCAHH     float64
AvgCAHA     float64
BFECAHH     float64
BFECAHA     float64
Length: 132, dtype: object

In [27]:
team_matches[
    team_matches['Team'].isin(['Man City', 'Man United'])
].groupby(['Team', 'Result']).size()

Team        Result
Man City    D          9
            L          6
            W         23
Man United  D         11
            L          7
            W         20
dtype: int64

In [28]:
team_matches['Points'] = team_matches['Result'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

team_matches['GoalDifference'] = (
    team_matches['Goals'] - team_matches['GoalsConceded']
)

league_table_python = team_matches.groupby('Team').agg(
    Played=('Team', 'count'),
    Wins=('Result', lambda x: (x == 'W').sum()),
    Draws=('Result', lambda x: (x == 'D').sum()),
    Losses=('Result', lambda x: (x == 'L').sum()),
    GoalsFor=('Goals', 'sum'),
    GoalsAgainst=('GoalsConceded', 'sum'),
    GoalDifference=('GoalDifference', 'sum'),
    Points=('Points', 'sum')
).reset_index()

league_table_python = league_table_python.sort_values(
    ['Points', 'GoalDifference'],
    ascending=[False, False]
).reset_index(drop=True)

league_table_python['Position'] = (
    league_table_python.index + 1
)

league_table_python

,Team,Played,Wins,Draws,Losses,GoalsFor,GoalsAgainst,GoalDifference,Points,Position
0,Arsenal,38,26,7,5,71,27,44,85,1
1,Man City,38,23,9,6,77,35,42,78,2
2,Man United,38,20,11,7,69,50,19,71,3
3,Aston Villa,38,19,8,11,56,49,7,65,4
4,Liverpool,38,17,9,12,63,53,10,60,5
5,Bournemouth,38,13,18,7,58,54,4,57,6
6,Sunderland,38,14,12,12,42,48,-6,54,7
7,Brighton,38,14,11,13,52,46,6,53,8
8,Brentford,38,14,11,13,55,52,3,53,9
9,Chelsea,38,14,10,14,58,52,6,52,10


In [29]:
team_matches.to_csv(
    "../data/processed/team_matches.csv",
    index=False
)

print("Saved corrected team_matches.csv")
print(team_matches.shape)

Saved corrected team_matches.csv
(760, 9)


In [30]:
league_table_python.to_csv(
    "../data/processed/league_table.csv",
    index=False
)

print("Saved corrected league_table.csv")
print(league_table_python.shape)

Saved corrected league_table.csv
(20, 10)


In [10]:
df.select_dtypes(include="string").columns.tolist()

['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTR', 'HTR', 'Referee']

In [11]:
df.isna().sum()

Div          0
Date         0
Time         0
HomeTeam     0
AwayTeam     0
            ..
MaxCAHA      0
AvgCAHH      0
AvgCAHA      0
BFECAHH     22
BFECAHA     22
Length: 132, dtype: int64

In [12]:
df.isna().sum()[df.isna().sum() > 0]

BFDH          1
BFDD          1
BFDA          1
BMGMH         2
BMGMD         2
BMGMA         2
BVH           2
BVD           2
BVA           2
CLH          98
CLD          98
CLA          98
LBH          94
LBD          94
LBA          94
PSH         170
PSD         170
PSA         170
BFEH         20
BFED         20
BFEA         20
P>2.5       170
P<2.5       170
BFE>2.5      20
BFE<2.5      20
AHh           1
PAHH        170
PAHA        170
BFEAHH       20
BFEAHA       20
BFDCH         8
BFDCD         8
BFDCA         8
BVCH          8
BVCD          8
BVCA          8
CLCH        120
CLCD        120
CLCA        120
LBCH         99
LBCD         99
LBCA         99
PSCH        170
PSCD        170
PSCA        170
BFECH        22
BFECD        22
BFECA        22
PC>2.5      170
PC<2.5      170
BFEC>2.5     22
BFEC<2.5     22
PCAHH       170
PCAHA       170
BFECAHH      22
BFECAHA      22
dtype: int64

In [13]:
df.describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HF,AF,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
count,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,...,380.000000,380.000000,210.000000,210.000000,380.000000,380.000000,380.000000,380.000000,358.000000,358.000000
mean,1.526316,1.223684,0.692105,0.497368,13.839474,11.157895,4.502632,3.871053,10.652632,10.989474,...,1.919368,1.930447,1.964762,1.966143,1.942947,1.956474,1.873974,1.886105,1.990782,1.998883
std,1.169914,1.084779,0.777277,0.742593,4.949137,4.573217,2.244311,2.112085,3.215295,3.397173,...,0.096736,0.095349,0.110116,0.111749,0.092333,0.090155,0.083938,0.084617,0.096875,0.097022
min,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,0.000000,0.000000,3.000000,3.000000,...,1.680000,1.650000,1.680000,1.680000,1.750000,1.750000,1.710000,1.700000,1.810000,1.740000
25%,1.000000,0.000000,0.000000,0.000000,11.000000,8.000000,3.000000,2.000000,8.000000,9.000000,...,1.850000,1.850000,1.880000,1.880000,1.870000,1.880000,1.810000,1.820000,1.910000,1.920000
50%,1.000000,1.000000,1.000000,0.000000,14.000000,11.000000,4.000000,4.000000,10.000000,11.000000,...,1.930000,1.930000,1.970000,1.950000,1.930000,1.950000,1.865000,1.890000,1.980000,2.000000
75%,2.000000,2.000000,1.000000,1.000000,16.250000,14.000000,6.000000,5.000000,13.000000,13.000000,...,2.000000,2.000000,2.040000,2.050000,2.030000,2.030000,1.940000,1.950000,2.060000,2.077500
max,5.000000,5.000000,3.000000,4.000000,35.000000,30.000000,11.000000,11.000000,20.000000,21.000000,...,2.200000,2.150000,2.320000,2.330000,2.350000,2.280000,2.100000,2.080000,2.320000,2.220000


In [14]:
football_cols = [
    'FTHG', 'FTAG',
    'HTHG', 'HTAG',
    'HS', 'AS',
    'HST', 'AST',
    'HF', 'AF',
    'HC', 'AC',
    'HY', 'AY',
    'HR', 'AR'
]

df[football_cols].describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
count,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000
mean,1.526316,1.223684,0.692105,0.497368,13.839474,11.157895,4.502632,3.871053,10.652632,10.989474,5.394737,4.602632,1.665789,2.081579,0.050000,0.052632
std,1.169914,1.084779,0.777277,0.742593,4.949137,4.573217,2.244311,2.112085,3.215295,3.397173,2.729101,2.748469,1.229105,1.318101,0.241204,0.223591
min,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,0.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,11.000000,8.000000,3.000000,2.000000,8.000000,9.000000,3.000000,3.000000,1.000000,1.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,0.000000,14.000000,11.000000,4.000000,4.000000,10.000000,11.000000,5.000000,4.000000,2.000000,2.000000,0.000000,0.000000
75%,2.000000,2.000000,1.000000,1.000000,16.250000,14.000000,6.000000,5.000000,13.000000,13.000000,7.000000,6.000000,2.000000,3.000000,0.000000,0.000000
max,5.000000,5.000000,3.000000,4.000000,35.000000,30.000000,11.000000,11.000000,20.000000,21.000000,14.000000,15.000000,5.000000,6.000000,2.000000,1.000000


In [15]:
df['FTR'].value_counts()

FTR
H    162
A    114
D    104
Name: count, dtype: int64

In [16]:
.value_counts()

SyntaxError: invalid syntax (747460836.py, line 1)

In [ ]:
df[['FTHG', 'FTAG']].mean()

In [ ]:
df[football_cols].mean()

In [ ]:
df.iloc[0]

In [ ]:
df[
    [
        'HomeTeam', 'AwayTeam',
        'FTHG', 'FTAG',
        'HS', 'AS',
        'HST', 'AST',
        'HC', 'AC',
        'HY', 'AY',
        'HR', 'AR',
        'FTR'
    ]
].head()

In [ ]:
home = df[
    [
        'HomeTeam', 'AwayTeam',
        'FTHG', 'FTAG',
        'HS', 'AS',
        'HST', 'AST',
        'HC', 'AC',
        'HY', 'HR',
        'FTR'
    ]
].copy()

In [ ]:
home = home.rename(
    columns={
        'HomeTeam': 'Team',
        'AwayTeam': 'Opponent',
        'FTHG': 'Goals',
        'FTAG': 'GoalsConceded',
        'HS': 'Shots',
        'AS': 'ShotsConceded',
        'HST': 'ShotsOnTarget',
        'AST': 'ShotsOnTargetConceded',
        'HC': 'Corners',
        'AC': 'CornersConceded',
        'HY': 'YellowCards',
        'HR': 'RedCards'
    }
)

In [ ]:
home['Venue'] = 'Home'

In [ ]:
home.head()

In [ ]:
away = df[
    [
        'AwayTeam', 'HomeTeam',
        'FTAG', 'FTHG',
        'AS', 'HS',
        'AST', 'HST',
        'AC', 'HC',
        'AY', 'AR',
        'FTR'
    ]
].copy()

In [ ]:
away.head()

In [ ]:
away = away.rename(
    columns={
        'AwayTeam': 'Team',
        'HomeTeam': 'Opponent',
        'FTAG': 'Goals',
        'FTHG': 'GoalsConceded',
        'AS': 'Shots',
        'HS': 'ShotsConceded',
        'AST': 'ShotsOnTarget',
        'HST': 'ShotsOnTargetConceded',
        'AC': 'Corners',
        'HC': 'CornersConceded',
        'AY': 'YellowCards',
        'AR': 'RedCards'
    }
)

In [ ]:
away['Venue'] = 'Away'

In [ ]:
away.head()

In [ ]:
team_matches = pd.concat([home, away], ignore_index=True)

In [ ]:
team_matches.shape

In [ ]:
def get_result(row):
    if row['FTR'] == 'D':
        return 'D'
    elif row['FTR'] == 'H' and row['Venue'] == 'Home':
        return 'W'
    elif row['FTR'] == 'A' and row['Venue'] == 'Away':
        return 'W'
    else:
        return 'L'

In [ ]:
team_matches['Result'] = team_matches.apply(get_result, axis=1)

In [ ]:
team_matches[['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head(10)

In [ ]:
team_matches[team_matches['Venue'] == 'Away'][['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head()

In [ ]:
team_matches[team_matches['Venue'] == 'Away'][['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head(1)

In [ ]:
team_matches['Result'].value_counts()

In [ ]:
276+276+208

In [ ]:
team_matches['Team'].value_counts().sort_index()

In [ ]:
team_matches['Points'] = team_matches['Result'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

In [ ]:
team_matches[['Team', 'Result', 'Points']].head(10)

In [ ]:
team_matches = team_matches.drop(columns=['FTR'])

In [ ]:
team_matches.head()

In [ ]:
team_matches['GoalDifference'] = (
    team_matches['Goals'] - team_matches['GoalsConceded']
)

In [ ]:
team_matches[['Team', 'Goals', 'GoalsConceded', 'GoalDifference']].head(10)

In [ ]:
(team_matches['Goals'] - team_matches['GoalsConceded'] == team_matches['GoalDifference']).all()

In [ ]:
team_matches.groupby('Team').size()

In [ ]:
team_matches['Result'].eq('W').sum()

In [ ]:
team_matches['Win'] = team_matches['Result'].eq('W')

In [ ]:
team_matches['Win'].head()

In [ ]:
team_matches.groupby('Team')['Win'].sum()

In [ ]:
team_matches.groupby('Team')['Win'].sum().sort_values(ascending=False)

In [ ]:
team_matches['Draw'] = team_matches['Result'].eq('D')

In [ ]:
team_matches['Draw'].head()

In [ ]:
team_matches.groupby('Team')['Draw'].sum().sort_values(ascending=False)

In [ ]:
team_matches['Loss'] = team_matches['Result'].eq('L')

In [ ]:
team_matches[['Team', 'Result', 'Win', 'Draw', 'Loss']].head(10)

In [ ]:
team_wins = team_matches.groupby('Team')['Win'].sum()

In [ ]:
team_wins

In [ ]:
team_draws = team_matches.groupby('Team')['Draw'].sum()

In [ ]:
team_draws

In [ ]:
team_losses = team_matches.groupby('Team')['Loss'].sum()

In [ ]:
team_losses

In [ ]:
league_table = pd.DataFrame({
    'Wins': team_wins,
    'Draws': team_draws,
    'Losses': team_losses
})

In [ ]:
league_table

In [ ]:
league_table['Played'] = (
    league_table['Wins']
    + league_table['Draws']
    + league_table['Losses']
)

In [ ]:
league_table

In [ ]:
team_goals = team_matches.groupby('Team')['Goals'].sum()

In [ ]:
team_goals

In [ ]:
team_goals_conceded = team_matches.groupby('Team')['GoalsConceded'].sum()

In [ ]:
team_goals_conceded

In [ ]:
team_goals.sum(), team_goals_conceded.sum()

In [ ]:
league_table['GoalsFor'] = team_goals
league_table['GoalsAgainst'] = team_goals_conceded

In [ ]:
league_table

In [ ]:
league_table['GoalDifference'] = (league_table['GoalsFor'] - league_table['GoalsAgainst'])

In [ ]:
league_table

In [ ]:
league_table['Points'] = (league_table['Wins'] * 3 + league_table['Draws'])

In [ ]:
league_table

In [ ]:
league_table = league_table.sort_values(['Points', 'GoalDifference', 'GoalsFor'], ascending=[False, False, False])

In [ ]:
league_table

In [ ]:
league_table['Position'] = range(1, len(league_table) + 1)

In [ ]:
league_table

In [ ]:
league_table = league_table[['Position', 'Played', 'Wins', 'Draws', 'Losses', 'GoalsFor', 'GoalsAgainst', 'GoalDifference', 'Points']]

In [ ]:
league_table

In [ ]:
(league_table['Wins'] + league_table['Losses'] + league_table['Draws'] == league_table['Played']).all()

In [ ]:
(
    league_table['Wins'] * 3
    + league_table['Draws']
    == league_table['Points']
).all()

In [ ]:
(
    league_table['GoalsFor']
    - league_table['GoalsAgainst']
    == league_table['GoalDifference']
).all()

In [ ]:
league_table.to_csv('../data/processed/league_table.csv', index=False)

In [ ]:
pd.read_csv('../data/processed/league_table.csv').head()

In [ ]:
league_table.columns

In [ ]:
league_table = league_table.reset_index()

In [ ]:
league_table.columns

In [ ]:
league_table = league_table[
    [
        'Position',
        'Team',
        'Played',
        'Wins',
        'Draws',
        'Losses',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'Points'
    ]
]

In [ ]:
league_table.head()

In [ ]:
league_table.to_csv(
    '../data/processed/league_table.csv',
    index=False
)

In [ ]:
saved_league_table = pd.read_csv(
    '../data/processed/league_table.csv'
)

saved_league_table.columns

In [ ]:
saved_league_table.equals(league_table)

In [ ]:
team_matches.to_csv(
    '../data/processed/team_matches.csv',
    index=False
)

In [ ]:
pd.read_csv(
    '../data/processed/team_matches.csv'
).shape

In [ ]:
team_matches.head()

In [ ]:
team_matches.groupby('Team')['Goals'].mean().sort_values(ascending=False)

In [ ]:
team_matches.groupby('Team')['GoalsConceded'].mean().sort_values()

In [ ]:
team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

In [ ]:
team_metrics = team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

In [ ]:
team_metrics['GoalDifferencePerMatch'] = (
    team_metrics['GoalsPerMatch']
    - team_metrics['GoalsConcededPerMatch']
)

In [ ]:
team_metrics.sort_values(
    'GoalDifferencePerMatch',
    ascending=False
)

In [ ]:
team_metrics.index

In [ ]:
league_table.index

In [ ]:
team_metrics = team_metrics.reset_index()

In [ ]:
team_metrics.head()

In [ ]:
team_performance = league_table.merge(
    team_metrics,
    on='Team',
    how='left'
)

In [ ]:
team_performance.head()

In [ ]:
team_performance.isna().sum()

In [ ]:
team_performance.to_csv(
    '../data/processed/team_performance.csv',
    index=False
)

In [ ]:
team_shot_metrics = team_matches.groupby('Team').agg(
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean')
)

team_shot_metrics.head()

In [ ]:
home_away_goals = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

home_away_goals.head(10)

In [ ]:
home_away_goals = home_away_goals.reset_index()

home_away_goals.head()

In [ ]:
home_away_comparison = home_away_goals.pivot(
    index='Team',
    columns='Venue',
    values=['GoalsPerMatch', 'GoalsConcededPerMatch']
)

home_away_comparison.head()

In [ ]:
home_away_comparison.columns = [
    'AwayGoalsPerMatch',
    'HomeGoalsPerMatch',
    'AwayGoalsConcededPerMatch',
    'HomeGoalsConcededPerMatch'
]

home_away_comparison = home_away_comparison.reset_index()

home_away_comparison.head()

In [ ]:
home_away_comparison['HomeGoalAdvantage'] = (
    home_away_comparison['HomeGoalsPerMatch']
    - home_away_comparison['AwayGoalsPerMatch']
)

home_away_comparison['HomeDefensiveAdvantage'] = (
    home_away_comparison['AwayGoalsConcededPerMatch']
    - home_away_comparison['HomeGoalsConcededPerMatch']
)

home_away_comparison.head()

In [ ]:
home_away_comparison.sort_values(
    'HomeGoalAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayGoalsPerMatch',
        'HomeGoalsPerMatch',
        'HomeGoalAdvantage'
    ]
]

In [ ]:
home_away_comparison.sort_values(
    'HomeDefensiveAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayGoalsConcededPerMatch',
        'HomeGoalsConcededPerMatch',
        'HomeDefensiveAdvantage'
    ]
]

In [ ]:
home_away_comparison['HomePerformanceDifference'] = (
    home_away_comparison['HomeGoalAdvantage']
    + home_away_comparison['HomeDefensiveAdvantage']
)

home_away_comparison.sort_values(
    'HomePerformanceDifference',
    ascending=False
)[
    [
        'Team',
        'HomeGoalAdvantage',
        'HomeDefensiveAdvantage',
        'HomePerformanceDifference'
    ]
]

In [ ]:
home_away_points = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    PointsPerMatch=('Points', 'mean')
)

home_away_points = home_away_points.reset_index()

home_away_points.head(10)

In [ ]:
home_away_points_comparison = home_away_points.pivot(
    index='Team',
    columns='Venue',
    values='PointsPerMatch'
)

home_away_points_comparison.columns = [
    'AwayPointsPerMatch',
    'HomePointsPerMatch'
]

home_away_points_comparison = home_away_points_comparison.reset_index()

home_away_points_comparison['HomePointsAdvantage'] = (
    home_away_points_comparison['HomePointsPerMatch']
    - home_away_points_comparison['AwayPointsPerMatch']
)

home_away_points_comparison.sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
team_matches.columns.tolist()

In [ ]:
saved_team_matches = pd.read_csv(
    '../data/processed/team_matches.csv'
)

saved_team_matches.columns.tolist()

In [ ]:
home_away_points_comparison.sort_values(
    'HomePointsAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayPointsPerMatch',
        'HomePointsPerMatch',
        'HomePointsAdvantage'
    ]
]

In [ ]:
home_away_analysis = home_away_comparison.merge(
    home_away_points_comparison,
    on='Team',
    how='inner'
)

home_away_analysis[
    [
        'Team',
        'HomeGoalAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    [
        'HomeGoalAdvantage',
        'HomePointsAdvantage'
    ]
].corr()

In [ ]:
home_away_shots = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean')
)

home_away_shots = home_away_shots.reset_index()
home_away_shots.head(10)

In [ ]:
home_away_shots_comparison = home_away_shots.pivot(
    index='Team',
    columns='Venue',
    values=[
        'ShotsPerMatch',
        'ShotsOnTargetPerMatch'
    ]
)

home_away_shots_comparison

In [ ]:
home_away_shots.columns.tolist()

In [ ]:
home_away_shots_comparison.columns = [
    'AwayShotsPerMatch',
    'HomeShotsPerMatch',
    'AwayShotsOnTargetPerMatch',
    'HomeShotsOnTargetPerMatch'
]

home_away_shots_comparison = home_away_shots_comparison.reset_index()

home_away_shots_comparison.head()

In [ ]:
home_away_shots_comparison['HomeShotsAdvantage'] = (
    home_away_shots_comparison['HomeShotsPerMatch']
    - home_away_shots_comparison['AwayShotsPerMatch']
)

home_away_shots_comparison['HomeShotsOnTargetAdvantage'] = (
    home_away_shots_comparison['HomeShotsOnTargetPerMatch']
    - home_away_shots_comparison['AwayShotsOnTargetPerMatch']
)

home_away_shots_comparison[
    [
        'Team',
        'HomeShotsAdvantage',
        'HomeShotsOnTargetAdvantage'
    ]
].sort_values(
    'HomeShotsAdvantage',
    ascending=False
)

In [ ]:
home_away_shots_comparison['AwayShotAccuracy'] = (
    home_away_shots_comparison['AwayShotsOnTargetPerMatch']
    / home_away_shots_comparison['AwayShotsPerMatch']
)

home_away_shots_comparison['HomeShotAccuracy'] = (
    home_away_shots_comparison['HomeShotsOnTargetPerMatch']
    / home_away_shots_comparison['HomeShotsPerMatch']
)

home_away_shots_comparison['HomeShotAccuracyAdvantage'] = (
    home_away_shots_comparison['HomeShotAccuracy']
    - home_away_shots_comparison['AwayShotAccuracy']
)

In [ ]:
home_away_shots_comparison[
    [
        'Team',
        'AwayShotAccuracy',
        'HomeShotAccuracy',
        'HomeShotAccuracyAdvantage'
    ]
].sort_values(
    'HomeShotAccuracyAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis = home_away_analysis.merge(
    home_away_shots_comparison[
        [
            'Team',
            'HomeShotsAdvantage',
            'HomeShotsOnTargetAdvantage',
            'HomeShotAccuracyAdvantage'
        ]
    ],
    on='Team',
    how='inner'
)

In [ ]:
home_away_analysis[
    [
        'HomeGoalAdvantage',
        'HomeShotsAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].corr()['HomePointsAdvantage'].sort_values(ascending=False)

In [ ]:
home_away_analysis.shape

In [ ]:
home_away_analysis.isna().sum()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    home_away_analysis['HomeShotAccuracyAdvantage'],
    home_away_analysis['HomePointsAdvantage']
)

plt.xlabel('Home Shot Accuracy Advantage')
plt.ylabel('Home Points Advantage')
plt.title('Home Shot Accuracy vs Home Points Advantage')

plt.axhline(0)
plt.axvline(0)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    home_away_analysis['HomeShotAccuracyAdvantage'],
    home_away_analysis['HomePointsAdvantage']
)

for _, row in home_away_analysis.iterrows():
    plt.annotate(
        row['Team'],
        (
            row['HomeShotAccuracyAdvantage'],
            row['HomePointsAdvantage']
        ),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('Home Shot Accuracy Advantage')
plt.ylabel('Home Points Advantage')
plt.title('Home Shot Accuracy vs Home Points Advantage')

plt.axhline(0)
plt.axvline(0)

plt.show()

In [ ]:
home_away_analysis[
    (home_away_analysis['HomeShotAccuracyAdvantage'] > 0) &
    (home_away_analysis['HomePointsAdvantage'] > 0)
][
    [
        'Team',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    (home_away_analysis['HomeShotAccuracyAdvantage'] < 0) &
    (home_away_analysis['HomePointsAdvantage'] > 0)
][
    [
        'Team',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    [
        'Team',
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_shots_conceded = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean')
)

home_away_shots_conceded = home_away_shots_conceded.reset_index()

In [ ]:
home_away_shots_conceded.head(10)

In [ ]:
home_away_shots_conceded_comparison = home_away_shots_conceded.pivot(
    index='Team',
    columns='Venue',
    values=[
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch'
    ]
)

home_away_shots_conceded_comparison

In [ ]:
home_away_shots_conceded_comparison.columns = [
    'AwayShotsConcededPerMatch',
    'HomeShotsConcededPerMatch',
    'AwayShotsOnTargetConcededPerMatch',
    'HomeShotsOnTargetConcededPerMatch'
]

home_away_shots_conceded_comparison = (
    home_away_shots_conceded_comparison
    .reset_index()
)

home_away_shots_conceded_comparison.head()

In [ ]:
home_away_shots_conceded_comparison['HomeShotsPreventionAdvantage'] = (
    home_away_shots_conceded_comparison['AwayShotsConcededPerMatch']
    - home_away_shots_conceded_comparison['HomeShotsConcededPerMatch']
)

home_away_shots_conceded_comparison['HomeShotsOnTargetPreventionAdvantage'] = (
    home_away_shots_conceded_comparison['AwayShotsOnTargetConcededPerMatch']
    - home_away_shots_conceded_comparison['HomeShotsOnTargetConcededPerMatch']
)

In [ ]:
home_away_shots_conceded_comparison.head()

In [ ]:
home_away_analysis = home_away_analysis.merge(
    home_away_shots_conceded_comparison[
        [
            'Team',
            'HomeShotsPreventionAdvantage',
            'HomeShotsOnTargetPreventionAdvantage'
        ]
    ],
    on='Team',
    how='inner'
)

In [ ]:
home_away_analysis.shape

In [ ]:
home_away_analysis[
    [
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotsPreventionAdvantage',
        'HomeShotsOnTargetPreventionAdvantage',
        'HomePointsAdvantage'
    ]
].corr()['HomePointsAdvantage'].sort_values(ascending=False)

In [ ]:
correlations = home_away_analysis[
    [
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotsPreventionAdvantage',
        'HomeShotsOnTargetPreventionAdvantage'
    ]
].corrwith(
    home_away_analysis['HomePointsAdvantage']
).sort_values(ascending=False)

correlations

In [ ]:
plt.figure(figsize=(10, 6))

correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Home Points Advantage')
plt.ylabel('Metric')
plt.title('Home Performance Metrics vs Home Points Advantage')
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Home Points Advantage')
plt.ylabel('Metric')
plt.title('Home Performance Metrics vs Home Points Advantage')
plt.xlim(-0.6, 0.6)
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
team_summary = team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean'),
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean'),
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean'),
    PointsPerMatch=('Points', 'mean')
).reset_index()

team_summary

In [ ]:
team_summary['ShotAccuracy'] = (
    team_summary['ShotsOnTargetPerMatch']
    / team_summary['ShotsPerMatch']
)

team_summary['OpponentShotAccuracy'] = (
    team_summary['ShotsOnTargetConcededPerMatch']
    / team_summary['ShotsConcededPerMatch']
)

team_summary

In [ ]:
team_correlations = team_summary[
    [
        'GoalsPerMatch',
        'GoalsConcededPerMatch',
        'ShotsPerMatch',
        'ShotsOnTargetPerMatch',
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch',
        'ShotAccuracy',
        'OpponentShotAccuracy'
    ]
].corrwith(
    team_summary['PointsPerMatch']
).sort_values(ascending=False)

team_correlations

In [ ]:
plt.figure(figsize=(10, 6))

team_correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Points Per Match')
plt.ylabel('Metric')
plt.title('Team Performance Metrics vs Points Per Match')
plt.xlim(-1, 1)
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
result_comparison = team_matches.groupby('Result').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean'),
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean'),
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean'),
    PointsPerMatch=('Points', 'mean')
).reindex(['W', 'D', 'L'])

result_comparison

In [ ]:
result_comparison['ShotAccuracy'] = (
    result_comparison['ShotsOnTargetPerMatch']
    / result_comparison['ShotsPerMatch']
)

result_comparison[['ShotAccuracy']]

In [ ]:
result_comparison['GoalConversion'] = (
    result_comparison['GoalsPerMatch']
    / result_comparison['ShotsPerMatch']
)

result_comparison[['GoalsPerMatch', 'ShotsPerMatch', 'GoalConversion']]

In [ ]:
result_comparison['GoalsPerShotOnTarget'] = (
    result_comparison['GoalsPerMatch']
    / result_comparison['ShotsOnTargetPerMatch']
)

result_comparison[['GoalsPerMatch', 'ShotsOnTargetPerMatch', 'GoalsPerShotOnTarget']]

In [ ]:
team_summary['GoalConversion'] = (
    team_summary['GoalsPerMatch']
    / team_summary['ShotsPerMatch']
)

team_summary[
    ['Team', 'GoalsPerMatch', 'ShotsPerMatch', 'GoalConversion', 'PointsPerMatch']
].sort_values('GoalConversion', ascending=False)

In [ ]:
team_summary[['GoalConversion', 'PointsPerMatch']].corr()

In [ ]:
defensive_correlations = team_summary[
    [
        'GoalsConcededPerMatch',
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch',
        'OpponentShotAccuracy'
    ]
].corrwith(
    team_summary['PointsPerMatch']
).sort_values()

defensive_correlations

In [ ]:
team_performance[
    [
        'Position',
        'Team',
        'Points',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'GoalsPerMatch',
        'GoalsConcededPerMatch',
        'PointsPerMatch'
    ]
]

In [ ]:
team_performance.columns.tolist()

In [ ]:
team_performance['PointsPerMatch'] = (
    team_performance['Points']
    / team_performance['Played']
)

In [ ]:
team_performance[
    [
        'Position',
        'Team',
        'Points',
        'PointsPerMatch',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'GoalsPerMatch',
        'GoalsConcededPerMatch'
    ]
]

In [ ]:
%whos

## Project Data Dictionary

### Core datasets

| Variable | Grain | Description |
|---|---|---|
| `df` | 1 row = match | Raw Premier League match dataset imported from Football-Data.co.uk |
| `team_matches` | 1 row = team-match | Match data transformed into a team perspective, giving each team one row per match |
| `league_table` | 1 row = team | Final league standings calculated from match results |
| `team_performance` | 1 row = team | League standings combined with goals-per-match and goals-conceded-per-match metrics |
| `team_summary` | 1 row = team | Team attacking, defensive and points-per-match metrics |
| `home_away_analysis` | 1 row = team | Comparison of each team's home and away performance |
| `result_comparison` | 1 row = result type | Comparison of match statistics across wins, draws and losses |

### Supporting / intermediate datasets

| Variable | Purpose |
|---|---|
| `home` | Home-team perspective before combining with away matches |
| `away` | Away-team perspective before combining with home matches |
| `home_away_goals` | Intermediate home/away goal metrics |
| `home_away_comparison` | Intermediate home/away goal comparison |
| `home_away_points` | Intermediate home/away points-per-match calculations |
| `home_away_points_comparison` | Home vs away points comparison |
| `home_away_shots` | Intermediate home/away shooting metrics |
| `home_away_shots_comparison` | Home vs away shooting comparison |
| `home_away_shots_conceded` | Intermediate defensive shooting metrics |
| `home_away_shots_conceded_comparison` | Home vs away defensive shooting comparison |

### Statistical outputs

| Variable | Description |
|---|---|
| `correlations` | Correlations between home-performance metrics and home points advantage |
| `team_correlations` | Correlations between team metrics and points per match |
| `defensive_correlations` | Correlations between defensive metrics and points per match |

### Helper objects

| Variable | Purpose |
|---|---|
| `football_cols` | List of core football-performance columns |
| `get_result` | Function used to convert match outcomes into W/D/L from each team's perspective |

## Machine Learning Preparation

The machine learning stage will use only information that would have been available before each match was played.

The prediction target is the match result:

- H = Home win
- D = Draw
- A = Away win

To avoid target leakage, match-level statistics from the match being predicted will not be used as input features.

In [33]:
df_2425 = pd.read_csv("../data/raw/E0_2425.csv")

df_2425.shape

(380, 120)

In [34]:
set(df.columns) - set(df_2425.columns)

{'BFDA',
 'BFDCA',
 'BFDCD',
 'BFDCH',
 'BFDD',
 'BFDH',
 'BMGMA',
 'BMGMCA',
 'BMGMCD',
 'BMGMCH',
 'BMGMD',
 'BMGMH',
 'BVA',
 'BVCA',
 'BVCD',
 'BVCH',
 'BVD',
 'BVH',
 'CLA',
 'CLCA',
 'CLCD',
 'CLCH',
 'CLD',
 'CLH',
 'LBA',
 'LBCA',
 'LBCD',
 'LBCH',
 'LBD',
 'LBH'}

In [35]:
set(df_2425.columns) - set(df.columns)

{'1XBA',
 '1XBCA',
 '1XBCD',
 '1XBCH',
 '1XBD',
 '1XBH',
 'BFA',
 'BFCA',
 'BFCD',
 'BFCH',
 'BFD',
 'BFH',
 'WHA',
 'WHCA',
 'WHCD',
 'WHCH',
 'WHD',
 'WHH'}

In [36]:
core_columns = [
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR',
    'HS',
    'AS',
    'HST',
    'AST',
    'HF',
    'AF',
    'HC',
    'AC',
    'HY',
    'AY',
    'HR',
    'AR'
]

[column for column in core_columns if column not in df_2425.columns]

[]

In [37]:
ml_columns = [
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR',
    'HS',
    'AS',
    'HST',
    'AST',
    'HF',
    'AF',
    'HC',
    'AC',
    'HY',
    'AY',
    'HR',
    'AR'
]

matches_2425 = df_2425[ml_columns].copy()
matches_2526 = df[ml_columns].copy()

matches_2425['Season'] = '2024/25'
matches_2526['Season'] = '2025/26'

matches_all = pd.concat(
    [matches_2425, matches_2526],
    ignore_index=True
)

matches_all.shape

(760, 19)

In [38]:
previous_season = matches_all[
    matches_all['Season'] == '2024/25'
].copy()

home_previous = previous_season[[
    'HomeTeam',
    'FTHG',
    'FTAG',
    'FTR'
]].copy()

home_previous.columns = [
    'Team',
    'GoalsFor',
    'GoalsAgainst',
    'Result'
]

away_previous = previous_season[[
    'AwayTeam',
    'FTAG',
    'FTHG',
    'FTR'
]].copy()

away_previous.columns = [
    'Team',
    'GoalsFor',
    'GoalsAgainst',
    'Result'
]

In [41]:
previous_team_matches = pd.concat(
    [home_previous, away_previous],
    ignore_index=True
)

previous_team_matches['Result'] = np.where(
    previous_team_matches['GoalsFor'] > previous_team_matches['GoalsAgainst'],
    'W',
    np.where(
        previous_team_matches['GoalsFor'] < previous_team_matches['GoalsAgainst'],
        'L',
        'D'
    )
)

previous_team_matches['Points'] = previous_team_matches['Result'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

previous_team_strength = previous_team_matches.groupby('Team').agg(
    PreviousPlayed=('Team', 'count'),
    PreviousWins=('Result', lambda x: (x == 'W').sum()),
    PreviousDraws=('Result', lambda x: (x == 'D').sum()),
    PreviousLosses=('Result', lambda x: (x == 'L').sum()),
    PreviousGoalsFor=('GoalsFor', 'sum'),
    PreviousGoalsAgainst=('GoalsAgainst', 'sum'),
    PreviousGoalDifference=('GoalsFor', lambda x: 0),
    PreviousPoints=('Points', 'sum')
).reset_index()

In [40]:
import numpy as np

In [42]:
previous_team_strength.head()

,Team,PreviousPlayed,PreviousWins,PreviousDraws,PreviousLosses,PreviousGoalsFor,PreviousGoalsAgainst,PreviousGoalDifference,PreviousPoints
0,Arsenal,38,20,14,4,69,34,0,74
1,Aston Villa,38,19,9,10,58,51,0,66
2,Bournemouth,38,15,11,12,58,46,0,56
3,Brentford,38,16,8,14,66,57,0,56
4,Brighton,38,16,13,9,66,59,0,61


In [43]:
previous_team_strength['PreviousGoalDifference'] = (
    previous_team_strength['PreviousGoalsFor']
    - previous_team_strength['PreviousGoalsAgainst']
)

previous_team_strength.head()

,Team,PreviousPlayed,PreviousWins,PreviousDraws,PreviousLosses,PreviousGoalsFor,PreviousGoalsAgainst,PreviousGoalDifference,PreviousPoints
0,Arsenal,38,20,14,4,69,34,35,74
1,Aston Villa,38,19,9,10,58,51,7,66
2,Bournemouth,38,15,11,12,58,46,12,56
3,Brentford,38,16,8,14,66,57,9,56
4,Brighton,38,16,13,9,66,59,7,61


In [44]:
previous_team_strength['PreviousPointsPerMatch'] = (
    previous_team_strength['PreviousPoints']
    / previous_team_strength['PreviousPlayed']
)

previous_team_strength.head()

,Team,PreviousPlayed,PreviousWins,PreviousDraws,PreviousLosses,PreviousGoalsFor,PreviousGoalsAgainst,PreviousGoalDifference,PreviousPoints,PreviousPointsPerMatch
0,Arsenal,38,20,14,4,69,34,35,74,1.947368
1,Aston Villa,38,19,9,10,58,51,7,66,1.736842
2,Bournemouth,38,15,11,12,58,46,12,56,1.473684
3,Brentford,38,16,8,14,66,57,9,56,1.473684
4,Brighton,38,16,13,9,66,59,7,61,1.605263


In [45]:
current_season_teams = set(
    matches_all.loc[
        matches_all['Season'] == '2025/26',
        ['HomeTeam', 'AwayTeam']
    ].stack()
)

previous_season_teams = set(
    previous_team_strength['Team']
)

promoted_teams = sorted(
    current_season_teams - previous_season_teams
)

promoted_teams

['Burnley', 'Leeds', 'Sunderland']

In [46]:
ml_matches = matches_all[
    matches_all['Season'] == '2025/26'
].copy()

ml_matches = ml_matches.sort_values(
    ['Date', 'HomeTeam', 'AwayTeam']
).reset_index(drop=True)

ml_matches.shape

(380, 19)

In [47]:
ml_matches = ml_matches.merge(
    previous_team_strength[
        [
            'Team',
            'PreviousPointsPerMatch',
            'PreviousGoalDifference'
        ]
    ],
    left_on='HomeTeam',
    right_on='Team',
    how='left'
)

ml_matches = ml_matches.rename(columns={
    'PreviousPointsPerMatch': 'HomePreviousPointsPerMatch',
    'PreviousGoalDifference': 'HomePreviousGoalDifference'
})

ml_matches = ml_matches.drop(columns='Team')

ml_matches.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,...,AF,HC,AC,HY,AY,HR,AR,Season,HomePreviousPointsPerMatch,HomePreviousGoalDifference
0,01/01/2026,Brentford,Tottenham,0,0,D,7,9,2,3,...,10,3,1,1,2,0,0,2025/26,1.473684,9.0
1,01/01/2026,Crystal Palace,Fulham,1,1,D,11,17,2,5,...,10,6,3,0,5,0,0,2025/26,1.394737,0.0
2,01/01/2026,Liverpool,Leeds,0,0,D,19,4,4,2,...,8,8,3,0,2,0,0,2025/26,2.210526,45.0
3,01/01/2026,Sunderland,Man City,0,0,D,10,14,4,4,...,6,3,5,1,1,0,0,2025/26,NaN,NaN
4,01/02/2026,Aston Villa,Brentford,0,1,A,27,6,5,2,...,8,12,1,0,3,0,0,2025/26,1.736842,7.0


In [48]:
ml_matches = ml_matches.merge(
    previous_team_strength[
        [
            'Team',
            'PreviousPointsPerMatch',
            'PreviousGoalDifference'
        ]
    ],
    left_on='AwayTeam',
    right_on='Team',
    how='left'
)

ml_matches = ml_matches.rename(columns={
    'PreviousPointsPerMatch': 'AwayPreviousPointsPerMatch',
    'PreviousGoalDifference': 'AwayPreviousGoalDifference'
})

ml_matches = ml_matches.drop(columns='Team')

ml_matches.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,...,AC,HY,AY,HR,AR,Season,HomePreviousPointsPerMatch,HomePreviousGoalDifference,AwayPreviousPointsPerMatch,AwayPreviousGoalDifference
0,01/01/2026,Brentford,Tottenham,0,0,D,7,9,2,3,...,1,1,2,0,0,2025/26,1.473684,9.0,1.000000,-1.0
1,01/01/2026,Crystal Palace,Fulham,1,1,D,11,17,2,5,...,3,0,5,0,0,2025/26,1.394737,0.0,1.421053,0.0
2,01/01/2026,Liverpool,Leeds,0,0,D,19,4,4,2,...,3,0,2,0,0,2025/26,2.210526,45.0,NaN,NaN
3,01/01/2026,Sunderland,Man City,0,0,D,10,14,4,4,...,5,1,1,0,0,2025/26,NaN,NaN,1.868421,28.0
4,01/02/2026,Aston Villa,Brentford,0,1,A,27,6,5,2,...,1,0,3,0,0,2025/26,1.736842,7.0,1.473684,9.0


In [49]:
ml_matches['Date'] = pd.to_datetime(
    ml_matches['Date'],
    dayfirst=True,
    errors='coerce'
)

ml_matches['Date'].isna().sum()

np.int64(0)

In [50]:
ml_matches['Date'].isna().sum()

np.int64(0)

In [51]:
ml_matches = ml_matches.sort_values(
    ['Date', 'HomeTeam', 'AwayTeam']
).reset_index(drop=True)

ml_matches[['Date', 'HomeTeam', 'AwayTeam']].head(10)

,Date,HomeTeam,AwayTeam
0,2025-08-15,Liverpool,Bournemouth
1,2025-08-16,Aston Villa,Newcastle
2,2025-08-16,Brighton,Fulham
3,2025-08-16,Sunderland,West Ham
4,2025-08-16,Tottenham,Burnley
5,2025-08-16,Wolves,Man City
6,2025-08-17,Chelsea,Crystal Palace
7,2025-08-17,Man United,Arsenal
8,2025-08-17,Nott'm Forest,Brentford
9,2025-08-18,Leeds,Everton


In [52]:
home_current = ml_matches[[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]].copy()

home_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

away_current = ml_matches[[
    'Date',
    'AwayTeam',
    'HomeTeam',
    'FTAG',
    'FTHG',
    'FTR'
]].copy()

away_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

current_team_matches = pd.concat(
    [home_current, away_current],
    ignore_index=True
)

current_team_matches = current_team_matches.sort_values(
    ['Team', 'Date']
).reset_index(drop=True)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult
0,2025-08-17,Arsenal,Man United,1,0,A
1,2025-08-23,Arsenal,Leeds,5,0,H
2,2025-08-31,Arsenal,Liverpool,0,1,H
3,2025-09-13,Arsenal,Nott'm Forest,3,0,H
4,2025-09-21,Arsenal,Man City,1,1,D
5,2025-09-28,Arsenal,Newcastle,2,1,A
6,2025-10-04,Arsenal,West Ham,2,0,H
7,2025-10-18,Arsenal,Fulham,1,0,A
8,2025-10-26,Arsenal,Crystal Palace,1,0,H
9,2025-11-01,Arsenal,Burnley,2,0,A


In [53]:
current_team_matches['Points'] = current_team_matches['MatchResult'].map({
    'H': 3,
    'D': 1,
    'A': 0
})

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points
0,2025-08-17,Arsenal,Man United,1,0,A,0
1,2025-08-23,Arsenal,Leeds,5,0,H,3
2,2025-08-31,Arsenal,Liverpool,0,1,H,3
3,2025-09-13,Arsenal,Nott'm Forest,3,0,H,3
4,2025-09-21,Arsenal,Man City,1,1,D,1
5,2025-09-28,Arsenal,Newcastle,2,1,A,0
6,2025-10-04,Arsenal,West Ham,2,0,H,3
7,2025-10-18,Arsenal,Fulham,1,0,A,0
8,2025-10-26,Arsenal,Crystal Palace,1,0,H,3
9,2025-11-01,Arsenal,Burnley,2,0,A,0


In [54]:
current_team_matches['MatchResult'] = np.where(
    current_team_matches['GoalsFor'] > current_team_matches['GoalsAgainst'],
    'W',
    np.where(
        current_team_matches['GoalsFor'] < current_team_matches['GoalsAgainst'],
        'L',
        'D'
    )
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points
0,2025-08-17,Arsenal,Man United,1,0,W,0
1,2025-08-23,Arsenal,Leeds,5,0,W,3
2,2025-08-31,Arsenal,Liverpool,0,1,L,3
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3
4,2025-09-21,Arsenal,Man City,1,1,D,1
5,2025-09-28,Arsenal,Newcastle,2,1,W,0
6,2025-10-04,Arsenal,West Ham,2,0,W,3
7,2025-10-18,Arsenal,Fulham,1,0,W,0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3
9,2025-11-01,Arsenal,Burnley,2,0,W,0


In [55]:
current_team_matches['Points'] = current_team_matches['MatchResult'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points
0,2025-08-17,Arsenal,Man United,1,0,W,3
1,2025-08-23,Arsenal,Leeds,5,0,W,3
2,2025-08-31,Arsenal,Liverpool,0,1,L,0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3
4,2025-09-21,Arsenal,Man City,1,1,D,1
5,2025-09-28,Arsenal,Newcastle,2,1,W,3
6,2025-10-04,Arsenal,West Ham,2,0,W,3
7,2025-10-18,Arsenal,Fulham,1,0,W,3
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3
9,2025-11-01,Arsenal,Burnley,2,0,W,3


In [56]:
current_team_matches['CumulativePoints'] = (
    current_team_matches
    .groupby('Team')['Points']
    .cumsum()
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints
0,2025-08-17,Arsenal,Man United,1,0,W,3,3
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9
4,2025-09-21,Arsenal,Man City,1,1,D,1,10
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25


In [57]:
current_team_matches['PreviousCumulativePoints'] = (
    current_team_matches
    .groupby('Team')['CumulativePoints']
    .shift(1)
    .fillna(0)
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints,PreviousCumulativePoints
0,2025-08-17,Arsenal,Man United,1,0,W,3,3,0.0
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6,3.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6,6.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9,6.0
4,2025-09-21,Arsenal,Man City,1,1,D,1,10,9.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13,10.0
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16,13.0
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19,16.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22,19.0
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25,22.0


In [58]:
current_team_matches['Previous5Points'] = (
    current_team_matches
    .groupby('Team')['Points']
    .shift(1)
    .rolling(5, min_periods=1)
    .sum()
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints,PreviousCumulativePoints,Previous5Points
0,2025-08-17,Arsenal,Man United,1,0,W,3,3,0.0,NaN
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6,3.0,3.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6,6.0,6.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9,6.0,6.0
4,2025-09-21,Arsenal,Man City,1,1,D,1,10,9.0,9.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13,10.0,10.0
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16,13.0,10.0
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19,16.0,10.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22,19.0,13.0
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25,22.0,13.0


In [59]:
current_team_matches['Previous5Points'] = (
    current_team_matches['Previous5Points']
    .fillna(0)
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints,PreviousCumulativePoints,Previous5Points
0,2025-08-17,Arsenal,Man United,1,0,W,3,3,0.0,0.0
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6,3.0,3.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6,6.0,6.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9,6.0,6.0
4,2025-09-21,Arsenal,Man City,1,1,D,1,10,9.0,9.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13,10.0,10.0
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16,13.0,10.0
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19,16.0,10.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22,19.0,13.0
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25,22.0,13.0


In [60]:
current_team_matches['Previous5GoalsFor'] = (
    current_team_matches
    .groupby('Team')['GoalsFor']
    .shift(1)
    .rolling(5, min_periods=1)
    .sum()
    .fillna(0)
)

current_team_matches['Previous5GoalsAgainst'] = (
    current_team_matches
    .groupby('Team')['GoalsAgainst']
    .shift(1)
    .rolling(5, min_periods=1)
    .sum()
    .fillna(0)
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints,PreviousCumulativePoints,Previous5Points,Previous5GoalsFor,Previous5GoalsAgainst
0,2025-08-17,Arsenal,Man United,1,0,W,3,3,0.0,0.0,0.0,0.0
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6,3.0,3.0,1.0,0.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6,6.0,6.0,6.0,0.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9,6.0,6.0,6.0,1.0
4,2025-09-21,Arsenal,Man City,1,1,D,1,10,9.0,9.0,9.0,1.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13,10.0,10.0,10.0,2.0
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16,13.0,10.0,11.0,3.0
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19,16.0,10.0,8.0,3.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22,19.0,13.0,9.0,2.0
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25,22.0,13.0,7.0,2.0


In [61]:
current_team_matches['Previous5Points'] = (
    current_team_matches
    .groupby('Team')['Points']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

current_team_matches['Previous5GoalsFor'] = (
    current_team_matches
    .groupby('Team')['GoalsFor']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

current_team_matches['Previous5GoalsAgainst'] = (
    current_team_matches
    .groupby('Team')['GoalsAgainst']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Points,CumulativePoints,PreviousCumulativePoints,Previous5Points,Previous5GoalsFor,Previous5GoalsAgainst
0,2025-08-17,Arsenal,Man United,1,0,W,3,3,0.0,0.0,0.0,0.0
1,2025-08-23,Arsenal,Leeds,5,0,W,3,6,3.0,3.0,1.0,0.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,0,6,6.0,6.0,6.0,0.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,3,9,6.0,6.0,6.0,1.0
4,2025-09-21,Arsenal,Man City,1,1,D,1,10,9.0,9.0,9.0,1.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,3,13,10.0,10.0,10.0,2.0
6,2025-10-04,Arsenal,West Ham,2,0,W,3,16,13.0,10.0,11.0,3.0
7,2025-10-18,Arsenal,Fulham,1,0,W,3,19,16.0,10.0,8.0,3.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,3,22,19.0,13.0,9.0,2.0
9,2025-11-01,Arsenal,Burnley,2,0,W,3,25,22.0,13.0,7.0,2.0


In [62]:
home_current = ml_matches[[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]].copy()

home_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

home_current['Venue'] = 'Home'


away_current = ml_matches[[
    'Date',
    'AwayTeam',
    'HomeTeam',
    'FTAG',
    'FTHG',
    'FTR'
]].copy()

away_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

away_current['Venue'] = 'Away'


current_team_matches = pd.concat(
    [home_current, away_current],
    ignore_index=True
)

current_team_matches = current_team_matches.sort_values(
    ['Team', 'Date']
).reset_index(drop=True)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Venue
0,2025-08-17,Arsenal,Man United,1,0,A,Away
1,2025-08-23,Arsenal,Leeds,5,0,H,Home
2,2025-08-31,Arsenal,Liverpool,0,1,H,Away
3,2025-09-13,Arsenal,Nott'm Forest,3,0,H,Home
4,2025-09-21,Arsenal,Man City,1,1,D,Home
5,2025-09-28,Arsenal,Newcastle,2,1,A,Away
6,2025-10-04,Arsenal,West Ham,2,0,H,Home
7,2025-10-18,Arsenal,Fulham,1,0,A,Away
8,2025-10-26,Arsenal,Crystal Palace,1,0,H,Home
9,2025-11-01,Arsenal,Burnley,2,0,A,Away


In [63]:
current_team_matches['MatchResult'] = np.where(
    current_team_matches['GoalsFor'] > current_team_matches['GoalsAgainst'],
    'W',
    np.where(
        current_team_matches['GoalsFor'] < current_team_matches['GoalsAgainst'],
        'L',
        'D'
    )
)

In [67]:
home_current = ml_matches[[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]].copy()

home_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

home_current['Venue'] = 'Home'


away_current = ml_matches[[
    'Date',
    'AwayTeam',
    'HomeTeam',
    'FTAG',
    'FTHG',
    'FTR'
]].copy()

away_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

away_current['Venue'] = 'Away'


current_team_matches = pd.concat(
    [home_current, away_current],
    ignore_index=True
)

current_team_matches = current_team_matches.sort_values(
    ['Team', 'Date']
).reset_index(drop=True)


current_team_matches['MatchResult'] = np.where(
    current_team_matches['GoalsFor'] > current_team_matches['GoalsAgainst'],
    'W',
    np.where(
        current_team_matches['GoalsFor'] < current_team_matches['GoalsAgainst'],
        'L',
        'D'
    )
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Venue
0,2025-08-17,Arsenal,Man United,1,0,W,Away
1,2025-08-23,Arsenal,Leeds,5,0,W,Home
2,2025-08-31,Arsenal,Liverpool,0,1,L,Away
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,Home
4,2025-09-21,Arsenal,Man City,1,1,D,Home
5,2025-09-28,Arsenal,Newcastle,2,1,W,Away
6,2025-10-04,Arsenal,West Ham,2,0,W,Home
7,2025-10-18,Arsenal,Fulham,1,0,W,Away
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,Home
9,2025-11-01,Arsenal,Burnley,2,0,W,Away


In [65]:
home_current['Venue'] = 'Home'

In [66]:
away_current['Venue'] = 'Away'

In [68]:
home_current[
    (home_current['Team'] == 'Arsenal') &
    (home_current['Opponent'] == 'Man United')
][['Date', 'Team', 'Opponent', 'Venue']]

,Date,Team,Opponent,Venue
225,2026-01-25,Arsenal,Man United,Home


In [69]:
current_team_matches[
    (current_team_matches['Team'] == 'Arsenal') &
    (current_team_matches['Opponent'] == 'Man United')
][['Date', 'Team', 'Opponent', 'MatchResult', 'Venue']]

,Date,Team,Opponent,MatchResult,Venue
0,2025-08-17,Arsenal,Man United,W,Away
22,2026-01-25,Arsenal,Man United,L,Home


In [70]:
home_current = ml_matches[[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]].copy()

home_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

home_current['Venue'] = 'Home'


away_current = ml_matches[[
    'Date',
    'AwayTeam',
    'HomeTeam',
    'FTAG',
    'FTHG',
    'FTR'
]].copy()

away_current.columns = [
    'Date',
    'Team',
    'Opponent',
    'GoalsFor',
    'GoalsAgainst',
    'MatchResult'
]

away_current['Venue'] = 'Away'


current_team_matches = pd.concat(
    [home_current, away_current],
    ignore_index=True
)

current_team_matches = current_team_matches.sort_values(
    ['Team', 'Date']
).reset_index(drop=True)


current_team_matches['MatchResult'] = np.where(
    current_team_matches['GoalsFor'] > current_team_matches['GoalsAgainst'],
    'W',
    np.where(
        current_team_matches['GoalsFor'] < current_team_matches['GoalsAgainst'],
        'L',
        'D'
    )
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Venue
0,2025-08-17,Arsenal,Man United,1,0,W,Away
1,2025-08-23,Arsenal,Leeds,5,0,W,Home
2,2025-08-31,Arsenal,Liverpool,0,1,L,Away
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,Home
4,2025-09-21,Arsenal,Man City,1,1,D,Home
5,2025-09-28,Arsenal,Newcastle,2,1,W,Away
6,2025-10-04,Arsenal,West Ham,2,0,W,Home
7,2025-10-18,Arsenal,Fulham,1,0,W,Away
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,Home
9,2025-11-01,Arsenal,Burnley,2,0,W,Away


In [71]:
current_team_matches[
    (current_team_matches['Team'] == 'Arsenal') &
    (current_team_matches['Opponent'] == 'Man United')
][['Date', 'Team', 'Opponent', 'MatchResult', 'Venue']]

,Date,Team,Opponent,MatchResult,Venue
0,2025-08-17,Arsenal,Man United,W,Away
22,2026-01-25,Arsenal,Man United,L,Home


In [72]:
ml_matches[
    (ml_matches['HomeTeam'] == 'Arsenal') &
    (ml_matches['AwayTeam'] == 'Man United')
][[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
225,2026-01-25,Arsenal,Man United,2,3,A


In [73]:
matches_2425 = df_2425[ml_columns].copy()
matches_2526 = df[ml_columns].copy()

matches_2425['Season'] = '2024/25'
matches_2526['Season'] = '2025/26'

matches_all = pd.concat(
    [matches_2425, matches_2526],
    ignore_index=True
)

ml_matches = matches_all[
    matches_all['Season'] == '2025/26'
].copy()

ml_matches['Date'] = pd.to_datetime(
    ml_matches['Date'],
    dayfirst=True,
    errors='coerce'
)

ml_matches = ml_matches.sort_values(
    ['Date', 'HomeTeam', 'AwayTeam']
).reset_index(drop=True)

ml_matches.shape

(380, 19)

In [74]:
ml_matches[
    (ml_matches['HomeTeam'] == 'Arsenal') &
    (ml_matches['AwayTeam'] == 'Man United')
][[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
225,2026-01-25,Arsenal,Man United,2,3,A


In [75]:
df[
    (df['HomeTeam'] == 'Arsenal') &
    (df['AwayTeam'] == 'Man United')
][[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR'
]]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
228,25/01/2026,Arsenal,Man United,2,3,A


In [76]:
df[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']].head(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,15/08/2025,Liverpool,Bournemouth,4,2,H
1,16/08/2025,Aston Villa,Newcastle,0,0,D
2,16/08/2025,Brighton,Fulham,1,1,D
3,16/08/2025,Sunderland,West Ham,3,0,H
4,16/08/2025,Tottenham,Burnley,3,0,H
5,16/08/2025,Wolves,Man City,0,4,A
6,17/08/2025,Chelsea,Crystal Palace,0,0,D
7,17/08/2025,Nott'm Forest,Brentford,3,1,H
8,17/08/2025,Man United,Arsenal,0,1,A
9,18/08/2025,Leeds,Everton,1,0,H


In [77]:
current_team_matches['Points'] = current_team_matches['MatchResult'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

current_team_matches['CumulativePoints'] = (
    current_team_matches
    .groupby('Team')['Points']
    .cumsum()
)

current_team_matches['PreviousCumulativePoints'] = (
    current_team_matches
    .groupby('Team')['CumulativePoints']
    .shift(1)
    .fillna(0)
)

current_team_matches['MatchesPlayedBefore'] = (
    current_team_matches
    .groupby('Team')
    .cumcount()
)

current_team_matches['Previous5Points'] = (
    current_team_matches
    .groupby('Team')['Points']
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
    .fillna(0)
)

current_team_matches['Previous5GoalsFor'] = (
    current_team_matches
    .groupby('Team')['GoalsFor']
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
    .fillna(0)
)

current_team_matches['Previous5GoalsAgainst'] = (
    current_team_matches
    .groupby('Team')['GoalsAgainst']
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
    .fillna(0)
)

current_team_matches.head(10)

,Date,Team,Opponent,GoalsFor,GoalsAgainst,MatchResult,Venue,Points,CumulativePoints,PreviousCumulativePoints,MatchesPlayedBefore,Previous5Points,Previous5GoalsFor,Previous5GoalsAgainst
0,2025-08-17,Arsenal,Man United,1,0,W,Away,3,3,0.0,0,0.0,0.0,0.0
1,2025-08-23,Arsenal,Leeds,5,0,W,Home,3,6,3.0,1,3.0,1.0,0.0
2,2025-08-31,Arsenal,Liverpool,0,1,L,Away,0,6,6.0,2,6.0,6.0,0.0
3,2025-09-13,Arsenal,Nott'm Forest,3,0,W,Home,3,9,6.0,3,6.0,6.0,1.0
4,2025-09-21,Arsenal,Man City,1,1,D,Home,1,10,9.0,4,9.0,9.0,1.0
5,2025-09-28,Arsenal,Newcastle,2,1,W,Away,3,13,10.0,5,10.0,10.0,2.0
6,2025-10-04,Arsenal,West Ham,2,0,W,Home,3,16,13.0,6,10.0,11.0,3.0
7,2025-10-18,Arsenal,Fulham,1,0,W,Away,3,19,16.0,7,10.0,8.0,3.0
8,2025-10-26,Arsenal,Crystal Palace,1,0,W,Home,3,22,19.0,8,13.0,9.0,2.0
9,2025-11-01,Arsenal,Burnley,2,0,W,Away,3,25,22.0,9,13.0,7.0,2.0


In [78]:
current_team_matches.groupby('Team').agg(
    Matches=('Team', 'count'),
    FirstMatchesPlayedBefore=('MatchesPlayedBefore', 'min'),
    LastMatchesPlayedBefore=('MatchesPlayedBefore', 'max')
)

,Matches,FirstMatchesPlayedBefore,LastMatchesPlayedBefore
Team,,,
Arsenal,38,0,37
Aston Villa,38,0,37
Bournemouth,38,0,37
Brentford,38,0,37
Brighton,38,0,37
Burnley,38,0,37
Chelsea,38,0,37
Crystal Palace,38,0,37
Everton,38,0,37


In [79]:
home_form = current_team_matches[
    current_team_matches['Venue'] == 'Home'
][[
    'Date',
    'Team',
    'PreviousCumulativePoints',
    'MatchesPlayedBefore',
    'Previous5Points',
    'Previous5GoalsFor',
    'Previous5GoalsAgainst'
]].copy()

home_form = home_form.rename(columns={
    'Team': 'HomeTeam',
    'PreviousCumulativePoints': 'HomePreviousCumulativePoints',
    'MatchesPlayedBefore': 'HomeMatchesPlayedBefore',
    'Previous5Points': 'HomePrevious5Points',
    'Previous5GoalsFor': 'HomePrevious5GoalsFor',
    'Previous5GoalsAgainst': 'HomePrevious5GoalsAgainst'
})

home_form.head()

,Date,HomeTeam,HomePreviousCumulativePoints,HomeMatchesPlayedBefore,HomePrevious5Points,HomePrevious5GoalsFor,HomePrevious5GoalsAgainst
1,2025-08-23,Arsenal,3.0,1,3.0,1.0,0.0
3,2025-09-13,Arsenal,6.0,3,6.0,6.0,1.0
4,2025-09-21,Arsenal,9.0,4,9.0,9.0,1.0
6,2025-10-04,Arsenal,13.0,6,10.0,11.0,3.0
8,2025-10-26,Arsenal,19.0,8,13.0,9.0,2.0


In [80]:
away_form = current_team_matches[
    current_team_matches['Venue'] == 'Away'
][[
    'Date',
    'Team',
    'PreviousCumulativePoints',
    'MatchesPlayedBefore',
    'Previous5Points',
    'Previous5GoalsFor',
    'Previous5GoalsAgainst'
]].copy()

away_form = away_form.rename(columns={
    'Team': 'AwayTeam',
    'PreviousCumulativePoints': 'AwayPreviousCumulativePoints',
    'MatchesPlayedBefore': 'AwayMatchesPlayedBefore',
    'Previous5Points': 'AwayPrevious5Points',
    'Previous5GoalsFor': 'AwayPrevious5GoalsFor',
    'Previous5GoalsAgainst': 'AwayPrevious5GoalsAgainst'
})

away_form.head()

,Date,AwayTeam,AwayPreviousCumulativePoints,AwayMatchesPlayedBefore,AwayPrevious5Points,AwayPrevious5GoalsFor,AwayPrevious5GoalsAgainst
0,2025-08-17,Arsenal,0.0,0,0.0,0.0,0.0
2,2025-08-31,Arsenal,6.0,2,6.0,6.0,0.0
5,2025-09-28,Arsenal,10.0,5,10.0,10.0,2.0
7,2025-10-18,Arsenal,16.0,7,10.0,8.0,3.0
9,2025-11-01,Arsenal,22.0,9,13.0,7.0,2.0


In [81]:
ml_dataset = ml_matches.merge(
    home_form,
    on=['Date', 'HomeTeam'],
    how='left'
)

ml_dataset = ml_dataset.merge(
    away_form,
    on=['Date', 'AwayTeam'],
    how='left'
)

ml_dataset.shape

(380, 29)

In [82]:
ml_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,...,HomePreviousCumulativePoints,HomeMatchesPlayedBefore,HomePrevious5Points,HomePrevious5GoalsFor,HomePrevious5GoalsAgainst,AwayPreviousCumulativePoints,AwayMatchesPlayedBefore,AwayPrevious5Points,AwayPrevious5GoalsFor,AwayPrevious5GoalsAgainst
0,2025-08-15,Liverpool,Bournemouth,4,2,H,19,10,10,3,...,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
1,2025-08-16,Aston Villa,Newcastle,0,0,D,3,16,3,3,...,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
2,2025-08-16,Brighton,Fulham,1,1,D,10,7,4,2,...,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
3,2025-08-16,Sunderland,West Ham,3,0,H,10,12,5,4,...,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
4,2025-08-16,Tottenham,Burnley,3,0,H,16,14,6,4,...,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0


In [83]:
ml_dataset['HomePromotedTeam'] = (
    ml_dataset['HomeTeam'].isin(promoted_teams).astype(int)
)

ml_dataset['AwayPromotedTeam'] = (
    ml_dataset['AwayTeam'].isin(promoted_teams).astype(int)
)

In [84]:
ml_dataset[
    (ml_dataset['HomePromotedTeam'] == 1) |
    (ml_dataset['AwayPromotedTeam'] == 1)
][[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'HomePreviousPointsPerMatch',
    'AwayPreviousPointsPerMatch',
    'HomePreviousGoalDifference',
    'AwayPreviousGoalDifference',
    'HomePromotedTeam',
    'AwayPromotedTeam'
]].head(10)

KeyError: "['HomePreviousPointsPerMatch', 'AwayPreviousPointsPerMatch', 'HomePreviousGoalDifference', 'AwayPreviousGoalDifference'] not in index"

In [85]:
ml_dataset.columns.tolist()

['Date',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR',
 'Season',
 'HomePreviousCumulativePoints',
 'HomeMatchesPlayedBefore',
 'HomePrevious5Points',
 'HomePrevious5GoalsFor',
 'HomePrevious5GoalsAgainst',
 'AwayPreviousCumulativePoints',
 'AwayMatchesPlayedBefore',
 'AwayPrevious5Points',
 'AwayPrevious5GoalsFor',
 'AwayPrevious5GoalsAgainst',
 'HomePromotedTeam',
 'AwayPromotedTeam']

In [86]:
ml_dataset = ml_dataset.merge(
    previous_team_strength[
        [
            'Team',
            'PreviousPointsPerMatch',
            'PreviousGoalDifference'
        ]
    ],
    left_on='HomeTeam',
    right_on='Team',
    how='left'
)

ml_dataset = ml_dataset.rename(columns={
    'PreviousPointsPerMatch': 'HomePreviousPointsPerMatch',
    'PreviousGoalDifference': 'HomePreviousGoalDifference'
})

ml_dataset = ml_dataset.drop(columns='Team')

In [87]:
ml_dataset = ml_dataset.merge(
    previous_team_strength[
        [
            'Team',
            'PreviousPointsPerMatch',
            'PreviousGoalDifference'
        ]
    ],
    left_on='AwayTeam',
    right_on='Team',
    how='left'
)

ml_dataset = ml_dataset.rename(columns={
    'PreviousPointsPerMatch': 'AwayPreviousPointsPerMatch',
    'PreviousGoalDifference': 'AwayPreviousGoalDifference'
})

ml_dataset = ml_dataset.drop(columns='Team')

In [88]:
ml_dataset[
    (ml_dataset['HomePromotedTeam'] == 1) |
    (ml_dataset['AwayPromotedTeam'] == 1)
][[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'HomePreviousPointsPerMatch',
    'AwayPreviousPointsPerMatch',
    'HomePreviousGoalDifference',
    'AwayPreviousGoalDifference',
    'HomePromotedTeam',
    'AwayPromotedTeam'
]].head(10)

,Date,HomeTeam,AwayTeam,HomePreviousPointsPerMatch,AwayPreviousPointsPerMatch,HomePreviousGoalDifference,AwayPreviousGoalDifference,HomePromotedTeam,AwayPromotedTeam
3,2025-08-16,Sunderland,West Ham,NaN,1.131579,NaN,-16.0,1,0
4,2025-08-16,Tottenham,Burnley,1.000000,NaN,-1.0,NaN,0,1
9,2025-08-18,Leeds,Everton,NaN,1.263158,NaN,-2.0,1,0
11,2025-08-23,Arsenal,Leeds,1.947368,NaN,35.0,NaN,0,1
14,2025-08-23,Burnley,Sunderland,NaN,NaN,NaN,NaN,1,1
21,2025-08-30,Leeds,Newcastle,NaN,1.736842,NaN,21.0,1,0
22,2025-08-30,Man United,Burnley,1.105263,NaN,-10.0,NaN,0,1
23,2025-08-30,Sunderland,Brentford,NaN,1.473684,NaN,9.0,1,0
33,2025-09-13,Crystal Palace,Sunderland,1.394737,NaN,0.0,NaN,0,1
35,2025-09-13,Fulham,Leeds,1.421053,NaN,0.0,NaN,0,1


In [89]:
ml_dataset['PreviousPPMDifference'] = (
    ml_dataset['HomePreviousPointsPerMatch']
    - ml_dataset['AwayPreviousPointsPerMatch']
)

ml_dataset['PreviousGDDifference'] = (
    ml_dataset['HomePreviousGoalDifference']
    - ml_dataset['AwayPreviousGoalDifference']
)

ml_dataset['Previous5PointsDifference'] = (
    ml_dataset['HomePrevious5Points']
    - ml_dataset['AwayPrevious5Points']
)

ml_dataset['Previous5GoalsForDifference'] = (
    ml_dataset['HomePrevious5GoalsFor']
    - ml_dataset['AwayPrevious5GoalsFor']
)

ml_dataset['Previous5GoalsAgainstDifference'] = (
    ml_dataset['HomePrevious5GoalsAgainst']
    - ml_dataset['AwayPrevious5GoalsAgainst']
)

In [90]:
ml_dataset[[
    'Date',
    'HomeTeam',
    'AwayTeam',
    'FTR',
    'PreviousPPMDifference',
    'PreviousGDDifference',
    'Previous5PointsDifference',
    'Previous5GoalsForDifference',
    'Previous5GoalsAgainstDifference',
    'HomePromotedTeam',
    'AwayPromotedTeam'
]].head(15)

,Date,HomeTeam,AwayTeam,FTR,PreviousPPMDifference,PreviousGDDifference,Previous5PointsDifference,Previous5GoalsForDifference,Previous5GoalsAgainstDifference,HomePromotedTeam,AwayPromotedTeam
0,2025-08-15,Liverpool,Bournemouth,H,0.736842,33.0,0.0,0.0,0.0,0,0
1,2025-08-16,Aston Villa,Newcastle,D,0.000000,-14.0,0.0,0.0,0.0,0,0
2,2025-08-16,Brighton,Fulham,D,0.184211,7.0,0.0,0.0,0.0,0,0
3,2025-08-16,Sunderland,West Ham,H,NaN,NaN,0.0,0.0,0.0,1,0
4,2025-08-16,Tottenham,Burnley,H,NaN,NaN,0.0,0.0,0.0,0,1
5,2025-08-16,Wolves,Man City,A,-0.763158,-43.0,0.0,0.0,0.0,0,0
6,2025-08-17,Chelsea,Crystal Palace,D,0.421053,21.0,0.0,0.0,0.0,0,0
7,2025-08-17,Man United,Arsenal,A,-0.842105,-45.0,0.0,0.0,0.0,0,0
8,2025-08-17,Nott'm Forest,Brentford,H,0.236842,3.0,0.0,0.0,0.0,0,0
9,2025-08-18,Leeds,Everton,H,NaN,NaN,0.0,0.0,0.0,1,0


In [91]:
ml_dataset[[
    'PreviousPPMDifference',
    'PreviousGDDifference'
]].isna().sum()

PreviousPPMDifference    108
PreviousGDDifference     108
dtype: int64

In [92]:
ml_dataset['ModelPreviousPPMDifference'] = (
    ml_dataset['PreviousPPMDifference'].fillna(0)
)

ml_dataset['ModelPreviousGDDifference'] = (
    ml_dataset['PreviousGDDifference'].fillna(0)
)

In [93]:
ml_dataset[[
    'PreviousPPMDifference',
    'ModelPreviousPPMDifference',
    'PreviousGDDifference',
    'ModelPreviousGDDifference',
    'HomePromotedTeam',
    'AwayPromotedTeam'
]].head(15)

,PreviousPPMDifference,ModelPreviousPPMDifference,PreviousGDDifference,ModelPreviousGDDifference,HomePromotedTeam,AwayPromotedTeam
0,0.736842,0.736842,33.0,33.0,0,0
1,0.000000,0.000000,-14.0,-14.0,0,0
2,0.184211,0.184211,7.0,7.0,0,0
3,NaN,0.000000,NaN,0.0,1,0
4,NaN,0.000000,NaN,0.0,0,1
5,-0.763158,-0.763158,-43.0,-43.0,0,0
6,0.421053,0.421053,21.0,21.0,0,0
7,-0.842105,-0.842105,-45.0,-45.0,0,0
8,0.236842,0.236842,3.0,3.0,0,0
9,NaN,0.000000,NaN,0.0,1,0


In [94]:
ml_dataset['MatchesPlayedDifference'] = (
    ml_dataset['HomeMatchesPlayedBefore']
    - ml_dataset['AwayMatchesPlayedBefore']
)

In [95]:
feature_columns = [
    'ModelPreviousPPMDifference',
    'ModelPreviousGDDifference',
    'Previous5PointsDifference',
    'Previous5GoalsForDifference',
    'Previous5GoalsAgainstDifference',
    'HomePromotedTeam',
    'AwayPromotedTeam',
    'MatchesPlayedDifference'
]

X = ml_dataset[feature_columns].copy()
y = ml_dataset['FTR'].copy()

X.shape, y.shape

((380, 8), (380,))

In [96]:
y.value_counts()

FTR
H    162
A    114
D    104
Name: count, dtype: int64

In [97]:
y.value_counts(normalize=True).round(3)

FTR
H    0.426
A    0.300
D    0.274
Name: proportion, dtype: float64

In [98]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((304, 8), (76, 8), (304,), (76,))

In [99]:
ml_dataset.loc[
    [split_index - 1, split_index],
    ['Date', 'HomeTeam', 'AwayTeam', 'FTR']
]

,Date,HomeTeam,AwayTeam,FTR
303,2026-03-21,Everton,Chelsea,H
304,2026-03-21,Fulham,Burnley,H


In [100]:
from sklearn.dummy import DummyClassifier

In [101]:
baseline_model = DummyClassifier(strategy='most_frequent')

In [102]:
baseline_model.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](3,)","[0.31,0.27,0.42]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](3,)","['A','D','H']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](8,)","['ModelPreviousPPMDifference','ModelPreviousGDDifference', 'Previous5PointsDifference',...,'HomePromotedTeam','AwayPromotedTeam', 'MatchesPlayedDifference']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,8
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [103]:
baseline_predictions = baseline_model.predict(X_test)

In [104]:
baseline_predictions[:10]

array(['H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H'], dtype='<U1')

In [105]:
from sklearn.metrics import accuracy_score

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

baseline_accuracy

0.4605263157894737

In [106]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [107]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [108]:
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['A','D','H']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['ModelPreviousPPMDifference','ModelPreviousGDDifference', 'Previous5PointsDifference',...,'HomePromotedTeam','AwayPromotedTeam', 'MatchesPlayedDifference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [109]:
logistic_predictions = logistic_model.predict(X_test)

In [110]:
logistic_predictions[:10]

array(['H', 'D', 'H', 'H', 'A', 'A', 'H', 'H', 'A', 'H'], dtype=object)

In [111]:
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

logistic_accuracy

0.4342105263157895

In [112]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, logistic_predictions)

array([[ 8,  0, 12],
       [ 6,  2, 13],
       [ 8,  4, 23]])

In [113]:
logistic_probabilities = logistic_model.predict_proba(X_test)

logistic_probabilities[:5]

array([[0.13232453, 0.38051268, 0.48716279],
       [0.34383107, 0.36820121, 0.28796771],
       [0.29362198, 0.21966277, 0.48671524],
       [0.0985276 , 0.28376561, 0.61770679],
       [0.48855359, 0.17972411, 0.3317223 ]])

In [114]:
logistic_model.classes_

array(['A', 'D', 'H'], dtype=object)

In [115]:
from sklearn.metrics import classification_report

print(classification_report(y_test, logistic_predictions))

              precision    recall  f1-score   support

           A       0.36      0.40      0.38        20
           D       0.33      0.10      0.15        21
           H       0.48      0.66      0.55        35

    accuracy                           0.43        76
   macro avg       0.39      0.38      0.36        76
weighted avg       0.41      0.43      0.40        76



In [116]:
ml_dataset.to_csv(
    "../data/processed/ml_dataset.csv",
    index=False
)